<figure>
    <center> <img src="../../images/mai_banner.png"  style="width:1000px;height:180px;" ></center>
</figure>

# Lab 09: Feature Engineering and Polynomial Regression

![](./images/C1_W2_Lab07_FeatureEngLecture.PNG)


## Objetivos

En este laboratorio vas a:
- Explorar la ingeniería de características y la regresión polinómica, lo que te permite utilizar la maquinaria de la regresión lineal para ajustar funciones muy complicadas, incluso muy no lineales.


## Herramientas
Utilizarás la función desarrollada en laboratorios anteriores, así como matplotlib y NumPy. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lab_utils_multi import zscore_normalize_features, run_gradient_descent_feng
np.set_printoptions(precision=2)  # reduced display precision on numpy arrays

<a name='FeatureEng'></a>
# Ingeniería de características y panorama de la regresión polinómica

Recordemos que la regresión lineal permite construir modelos de la forma:
$$f_{\mathbf{w},b}(x) = w_0x_0 + w_1x_1 + \cdots + w_{n-1}x_{n-1} + b \tag{1}$$

**¿Qué ocurre si las características o los datos no son lineales, o si son combinaciones de características?** Por ejemplo, los precios de las viviendas no suelen variar linealmente con la superficie habitable; las viviendas muy pequeñas o muy grandes se penalizan. 

¿Cómo podemos usar los mecanismos de la regresión lineal para ajustar una curva como esta? Recuerda que esos mecanismos nos permiten modificar los parámetros $\mathbf{w}$ y $b$ de (1) para ajustar la ecuación a los datos de entrenamiento. Sin embargo, por mucho que ajustemos $\mathbf{w}$ y $b$ en (1), no lograremos ajustar una curva no lineal.


<a name='PolynomialFeatures'></a>
## Características polinómicas

Hasta ahora hemos considerado un escenario en el que los datos no eran lineales. Probemos a usar lo que sabemos para ajustar una curva no lineal. Empezaremos con una función cuadrática sencilla: $y = 1+x^2$

Ya conoces todas las funciones que estamos usando. Puedes consultarlas en el archivo lab_utils.py. Usaremos [`np.c_[..]`](https://numpy.org/doc/stable/reference/generated/numpy.c_.html), una función de NumPy que concatena por columnas.

In [ ]:
# create target data
x = np.arange(0, 20, 1)
y = 1 + x**2
# Cambiamos a una matriz columna para que sea compatible con la regresión lineal multivariable
X = x.reshape(-1, 1)

model_w,model_b = run_gradient_descent_feng(X,y,iterations=1000, alpha = 1e-2)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("no feature engineering")
plt.plot(x,X@model_w + model_b, label="Predicted Value");  plt.xlabel("X"); plt.ylabel("y"); plt.legend(); plt.show()

Bueno, como era de esperar, el ajuste no es muy bueno. Lo que se necesita es algo como $y= w_0x_0^2 + b$, es decir, una **característica polinómica**.
Para lograrlo, puedes modificar los *datos de entrada* y crear las características necesarias. Si sustituyes los datos originales por una versión en la que el valor de $x$ está elevado al cuadrado, puedes obtener $y= w_0x_0^2 + b$. Probémoslo. A continuación, sustituye `X` por `X**2`:

In [ ]:
# Crear datos de destino
x = np.arange(0, 20, 1)
y = 1 + x**2

# Ingeniería de características
X = x**2      #<-- agregando ingenieria de caracteristicas

In [ ]:
X = X.reshape(-1, 1)  #X debe ser una matriz 2-D o vector columna
model_w,model_b = run_gradient_descent_feng(X, y, iterations=10000, alpha = 1e-5)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("Added x**2 feature")
plt.plot(x, np.dot(X,model_w) + model_b, label="Predicted Value"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

¡Excelente! El ajuste es casi perfecto. Observa los valores de $\mathbf{w}$ y b que aparecen justo encima del gráfico: `w,b found by gradient descent: w: [1.], b: 0.0490`. El descenso por gradiente modificó los valores iniciales de $\mathbf{w},b $ para obtener (1.0, 0.049), es decir, un modelo de $y=1*x_0^2+0.049$, muy cercano al valor objetivo de $y=1*x_0^2+1$. Si ejecutaras el algoritmo durante más tiempo, el ajuste podría ser aún mejor.

### Selección de Características

Antes, sabíamos que se necesitaba un término $x^2$. No siempre es evidente qué características se necesitan. Se pueden agregar varias características posibles para encontrar las más útiles. Por ejemplo, ¿qué habría pasado si hubiéramos probado: $y=w_0x_0 + w_1x_1^2 + w_2x_2^3+b$?

Ejecuta las siguientes celdas.

In [ ]:
# create target data
x = np.arange(0, 20, 1)
y = x**2

# engineer features .
X = np.c_[x, x**2, x**3]   #<-- added engineered feature

In [ ]:
model_w,model_b = run_gradient_descent_feng(X, y, iterations=10000, alpha=1e-7)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("x, x**2, x**3 features")
plt.plot(x, X@model_w + model_b, label="Predicted Value"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

Observa el valor de $\mathbf{w}$, `[0.08 0.54 0.03]`, y el de b, `0.0106`. Esto significa que el modelo después del ajuste o entrenamiento es:
$$ 0.08x + 0.54x^2 + 0.03x^3 + 0.0106 $$
El descenso por gradiente ha dado más importancia a la característica $x^2$, que es la que mejor se ajusta a los datos, al aumentar el término $w_1$ en relación con los demás. Si ejecutaras el algoritmo durante mucho más tiempo, seguiría reduciendo el impacto de los otros términos.
>El descenso por gradiente selecciona las características «correctas» al dar más importancia a sus parámetros asociados.

Repasemos esta idea:
- Un peso de menor valor indica que la característica es menos importante o adecuada. En el caso extremo, si el peso llega a cero o se acerca mucho a cero, la característica asociada no resulta útil para ajustar el modelo a los datos.
- En el ejemplo anterior, después del ajuste, el peso asociado a la característica $x^2$ es mucho mayor que los pesos de $x$ o $x^3$, porque es la característica más útil para ajustar el modelo a los datos.

### Otra perspectiva
Antes, las características polinómicas se eligieron según qué tan bien se ajustaban a los datos objetivo. Otra forma de verlo es recordar que seguimos usando regresión lineal una vez que creamos nuevas características. Por eso, las mejores características serán aquellas que tengan una relación lineal con el objetivo. Un ejemplo ayudará a entenderlo mejor.

In [ ]:
# create target data
x = np.arange(0, 20, 1)
y = x**2

# engineer features .
X = np.c_[x, x**2, x**3]   #<-- added engineered feature
X_features = ['x','x^2','x^3']

In [ ]:
fig,ax=plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X[:,i],y)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("y")
plt.show()

Como se observa arriba, la característica $x^2$ representada frente al valor objetivo $y$ tiene una relación lineal. Por lo tanto, la regresión lineal puede generar fácilmente un modelo usando esa característica.

### Escalado de características

Como se describió en el último laboratorio, si el conjunto de datos tiene características con escalas significativamente diferentes, se debe aplicar el escalado de características para acelerar el descenso de gradiente. En el ejemplo anterior, hay $x$, $x^2$ y $x^3$ que naturalmente tendrán escalas muy diferentes. Apliquemos la normalización Z-score a nuestro ejemplo.

In [ ]:
# create target data
x = np.arange(0,20,1)
X = np.c_[x, x**2, x**3]
print(f"Peak to Peak range by column in Raw        X:{np.ptp(X,axis=0)}")

# add mean_normalization 
X = zscore_normalize_features(X)     
print(f"Peak to Peak range by column in Normalized X:{np.ptp(X,axis=0)}")

Ahora podemos intentar de nuevo con un valor más agresivo de alpha:

In [ ]:
x = np.arange(0,20,1)
y = x**2

X = np.c_[x, x**2, x**3]
X = zscore_normalize_features(X) 

model_w, model_b = run_gradient_descent_feng(X, y, iterations=100000, alpha=1e-1)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("Normalized x x**2, x**3 feature")
plt.plot(x,X@model_w + model_b, label="Predicted Value"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

El escalado de características permite que esto converja mucho más rápido.
Observe nuevamente los valores de $\mathbf{w}$. El término $w_1$, que es el término $x^2$, es el más enfatizado. El descenso de gradiente ha prácticamente eliminado el término $x^3$.

### Funciones Complejas
Con la ingeniería de características, incluso funciones bastante complejas se pueden modelar:

In [ ]:
x = np.arange(0,20,1)
y = np.cos(x/2)

X = np.c_[x, x**2, x**3,x**4, x**5, x**6, x**7, x**8, x**9, x**10, x**11, x**12, x**13]
X = zscore_normalize_features(X) 

model_w,model_b = run_gradient_descent_feng(X, y, iterations=1000000, alpha = 1e-1)

plt.scatter(x, y, marker='x', c='r', label="Actual Value"); plt.title("Normalized x x**2, x**3 feature")
plt.plot(x,X@model_w + model_b, label="Predicted Value"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()



## ¡Felicidades!
- En este laboratorio aprendiste:
- aprendiste cómo la regresión lineal puede modelar funciones complejas, incluso altamente no lineales, utilizando la ingeniería de características
- reconociste que es importante aplicar el escalado de características al realizar ingeniería de características